# Phase 6 - Model Training on Segmented Images

Train SVM and Decision Tree models on images containing only important regions (from Phase 5).

## Section 1: Import Libraries and Configuration

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from skimage.feature import graycomatrix, graycoprops
from skimage.measure import shannon_entropy
from scipy.stats import skew, kurtosis
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.pipeline import Pipeline
from scipy.stats import loguniform
import joblib
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path("Dataset/Animals-10-Segmented")
OUTPUT_CSV = Path("CSV/features_segmented.csv")

CLASSES = [
    "butterfly", "cat", "chicken", "cow", "dog",
    "elephant", "horse", "sheep", "spider", "squirrel"
]

PROCESS_SIZE = (128, 128)
GLCM_DISTANCES = [1]
GLCM_ANGLES = [0, np.pi/4, np.pi/2, 3*np.pi/4]

print("Libraries imported successfully")
print(f"Dataset directory: {DATA_DIR}")
print(f"Number of classes: {len(CLASSES)}")

## Section 2: Feature Extraction Functions

In [ ]:
def compute_rgb_stats(img):
    b, g, r = cv2.split(img.astype(np.float32))
    return r.mean(), g.mean(), b.mean(), r.std(), g.std(), b.std()

def compute_hsv_stats(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
    h, s, v = cv2.split(hsv)
    return h.mean(), s.mean(), v.mean(), h.std(), s.std(), v.std()

def compute_intensity_stats(gray):
    g = gray.astype(np.float32)
    mean_int = g.mean()
    std_int = g.std()
    min_int = g.min()
    max_int = g.max()
    range_int = max_int - min_int
    flat = g.ravel()
    skewness_val = float(skew(flat, bias=False)) if flat.size > 1 else 0.0
    kurtosis_val = float(kurtosis(flat, fisher=True, bias=False)) if flat.size > 1 else 0.0
    entropy_val = float(shannon_entropy(gray))
    return mean_int, std_int, min_int, max_int, range_int, skewness_val, kurtosis_val, entropy_val

def compute_glcm_features(gray):
    gray_u8 = gray.astype(np.uint8)
    glcm = graycomatrix(gray_u8, distances=GLCM_DISTANCES, angles=GLCM_ANGLES,
                        levels=256, symmetric=True, normed=True)
    props = ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]
    vals = [float(graycoprops(glcm, p).mean()) for p in props]
    return vals

def compute_edge_features(gray):
    edges = cv2.Canny(gray, 100, 200)
    edge_density = float(np.count_nonzero(edges)) / edges.size
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    mag = np.sqrt(gx ** 2 + gy ** 2)
    mask = edges > 0
    if np.any(mask):
        vals = mag[mask]
        edge_mean = float(vals.mean())
        edge_std = float(vals.std())
    else:
        edge_mean = 0.0
        edge_std = 0.0
    return edge_mean, edge_std, edge_density

def compute_contour_features(gray):
    edges = cv2.Canny(gray, 100, 200)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return 0.0, 0.0, 0.0
    cnt = max(contours, key=cv2.contourArea)
    area = float(cv2.contourArea(cnt))
    perimeter = float(cv2.arcLength(cnt, True))
    compactness = (perimeter ** 2) / (4 * np.pi * area) if area > 0 else 0.0
    return area, perimeter, compactness

def extract_all_features(image_path):
    img = cv2.imread(str(image_path))
    if img is None:
        return None
    
    h, w = img.shape[:2]
    width = float(w)
    height = float(h)
    aspect_ratio = width / height if height > 0 else 0.0
    
    img_resized = cv2.resize(img, PROCESS_SIZE)
    gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
    
    mean_r, mean_g, mean_b, std_r, std_g, std_b = compute_rgb_stats(img_resized)
    mean_h, mean_s, mean_v, std_h, std_s, std_v = compute_hsv_stats(img_resized)
    brightness = mean_v
    contrast_simple = np.std(gray.astype(np.float32))
    
    mean_int, std_int, min_int, max_int, range_int, skewness_val, kurtosis_val, entropy_val = compute_intensity_stats(gray)
    contrast, dissimilarity, homogeneity, energy, correlation, ASM = compute_glcm_features(gray)
    edge_mean, edge_std, edge_density = compute_edge_features(gray)
    contour_area, contour_perimeter, compactness = compute_contour_features(gray)
    
    feats = np.array([
        width, height, aspect_ratio,
        mean_r, mean_g, mean_b, std_r, std_g, std_b,
        mean_h, mean_s, mean_v, std_h, std_s, std_v,
        brightness, contrast_simple,
        mean_int, std_int, min_int, max_int, range_int,
        skewness_val, kurtosis_val, entropy_val,
        contrast, dissimilarity, homogeneity, energy, correlation, ASM,
        edge_mean, edge_std, edge_density,
        contour_area, contour_perimeter, compactness
    ], dtype=np.float32)
    
    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)
    return feats

print("Feature extraction functions defined")

## Section 3: Extract Features from Segmented Images

In [ ]:
from tqdm import tqdm

data = []

print("Extracting features from segmented images...")

for class_name in CLASSES:
    class_dir = DATA_DIR / class_name
    
    if not class_dir.exists():
        print(f"Warning: {class_dir} does not exist")
        continue
    
    image_files = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.png"))
    
    print(f"\nProcessing class: {class_name} ({len(image_files)} images)")
    
    for img_path in tqdm(image_files, desc=class_name):
        features = extract_all_features(img_path)
        
        if features is not None:
            data.append({
                'filename': img_path.name,
                'label': class_name,
                'features': features
            })

print(f"\nTotal images processed: {len(data)}")

feature_matrix = np.array([d['features'] for d in data])
labels = [d['label'] for d in data]
filenames = [d['filename'] for d in data]

print(f"Feature matrix shape: {feature_matrix.shape}")

## Section 4: Save Features to CSV

In [ ]:
df_features = pd.DataFrame(feature_matrix)
df_features['filename'] = filenames
df_features['label'] = labels

cols = ['filename', 'label'] + [col for col in df_features.columns if col not in ['filename', 'label']]
df_features = df_features[cols]

OUTPUT_CSV.parent.mkdir(exist_ok=True)
df_features.to_csv(OUTPUT_CSV, index=False)

print(f"\nFeatures saved to {OUTPUT_CSV}")
print(f"Dataset shape: {df_features.shape}")
print(f"\nFirst few rows:")
print(df_features.head())

## Section 5: Prepare Data for Training

In [ ]:
y = df_features['label']
X = df_features.drop(columns=['filename', 'label'])

print(f"Feature matrix shape: {X.shape}")
print(f"Labels shape: {y.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nTrain/Test split completed")
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

## Section 6: SVM Training and Hyperparameter Tuning

In [ ]:
print("="*70)
print("SVM HYPERPARAMETER TUNING - RandomizedSearchCV")
print("="*70)

svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(random_state=42)),
])

svm_param_distributions = {
    "svm__C": loguniform(1e-3, 1e3),
    "svm__gamma": loguniform(1e-2, 1),
    "svm__kernel": ["rbf"],
}

print(f"\nSearch Strategy:")
print(f"   Method: RandomizedSearchCV")
print(f"   Iterations: 25 random combinations")
print(f"   Cross-validation: 5-fold")

print(f"\nParameters to tune:")
print(f"   C: loguniform(1e-3, 1e3)")
print(f"   kernel: rbf")
print(f"   gamma: loguniform(1e-2, 1)")

print(f"\nStarting tuning at {datetime.now().strftime('%H:%M:%S')}...")
start_time = time.time()

svm_random = RandomizedSearchCV(
    estimator=svm_pipeline,
    param_distributions=svm_param_distributions,
    n_iter=25,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2,
    random_state=42,
    refit=True
)

svm_random.fit(X_train, y_train)

elapsed_time = time.time() - start_time

print(f"\nTuning completed in {elapsed_time/60:.2f} minutes ({elapsed_time:.1f} seconds)")
print(f"Best SVM parameters: {svm_random.best_params_}")
print(f"Best cross-validation accuracy: {svm_random.best_score_:.4f}")

best_svm = svm_random.best_estimator_

## Section 7: SVM Model Evaluation

In [ ]:
print("\n" + "="*70)
print("SVM MODEL PERFORMANCE EVALUATION")
print("="*70)

svm_train_pred = best_svm.predict(X_train)
svm_test_pred = best_svm.predict(X_test)

train_acc = accuracy_score(y_train, svm_train_pred)
test_acc = accuracy_score(y_test, svm_test_pred)

train_precision, train_recall, train_f1, _ = precision_recall_fscore_support(y_train, svm_train_pred, average='weighted')
test_precision, test_recall, test_f1, _ = precision_recall_fscore_support(y_test, svm_test_pred, average='weighted')

print(f"\nTRAIN SET Performance:")
print(f"   Accuracy:  {train_acc:.4f}")
print(f"   Precision: {train_precision:.4f}")
print(f"   Recall:    {train_recall:.4f}")
print(f"   F1-Score:  {train_f1:.4f}")

print(f"\nTEST SET Performance:")
print(f"   Accuracy:  {test_acc:.4f}")
print(f"   Precision: {test_precision:.4f}")
print(f"   Recall:    {test_recall:.4f}")
print(f"   F1-Score:  {test_f1:.4f}")

overfit_gap = train_acc - test_acc
print(f"\nOverfitting Analysis:")
print(f"   Train-Test Gap: {overfit_gap:.4f}")
if overfit_gap < 0.05:
    print(f"   Status: Good generalization (gap < 5%)")
elif overfit_gap < 0.10:
    print(f"   Status: Slight overfitting (gap 5-10%)")
else:
    print(f"   Status: Significant overfitting (gap > 10%)")

print(f"\nDetailed Test Set Classification Report:")
print(classification_report(y_test, svm_test_pred, target_names=CLASSES))

## Section 8: Decision Tree Training and Hyperparameter Tuning

In [ ]:
print("\n" + "="*70)
print("DECISION TREE HYPERPARAMETER TUNING - RandomizedSearchCV")
print("="*70)

dt_param_distributions = {
    "max_depth": [10, 15, 20, 25, None],
    "criterion": ["gini", "entropy"],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None],
}

print(f"\nSearch Strategy:")
print(f"   Method: RandomizedSearchCV")
print(f"   Iterations: 10 random combinations")
print(f"   Cross-validation: 3-fold")

print(f"\nParameters to tune:")
print(f"   max_depth: {dt_param_distributions['max_depth']}")
print(f"   criterion: {dt_param_distributions['criterion']}")
print(f"   min_samples_split: {dt_param_distributions['min_samples_split']}")
print(f"   min_samples_leaf: {dt_param_distributions['min_samples_leaf']}")
print(f"   max_features: {dt_param_distributions['max_features']}")

print(f"\nStarting tuning at {datetime.now().strftime('%H:%M:%S')}...")
start_time = time.time()

dt_random = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_distributions=dt_param_distributions,
    n_iter=10,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2,
    random_state=42,
    refit=True
)

dt_random.fit(X_train, y_train)

elapsed_time = time.time() - start_time

print(f"\nTuning completed in {elapsed_time/60:.2f} minutes ({elapsed_time:.1f} seconds)")
print(f"Best Decision Tree parameters: {dt_random.best_params_}")
print(f"Best cross-validation accuracy: {dt_random.best_score_:.4f}")

best_dt = dt_random.best_estimator_

## Section 9: Decision Tree Model Evaluation

In [ ]:
print("\n" + "="*70)
print("DECISION TREE MODEL PERFORMANCE EVALUATION")
print("="*70)

dt_train_pred = best_dt.predict(X_train)
dt_test_pred = best_dt.predict(X_test)

train_acc_dt = accuracy_score(y_train, dt_train_pred)
test_acc_dt = accuracy_score(y_test, dt_test_pred)

train_precision_dt, train_recall_dt, train_f1_dt, _ = precision_recall_fscore_support(y_train, dt_train_pred, average='weighted')
test_precision_dt, test_recall_dt, test_f1_dt, _ = precision_recall_fscore_support(y_test, dt_test_pred, average='weighted')

print(f"\nTRAIN SET Performance:")
print(f"   Accuracy:  {train_acc_dt:.4f}")
print(f"   Precision: {train_precision_dt:.4f}")
print(f"   Recall:    {train_recall_dt:.4f}")
print(f"   F1-Score:  {train_f1_dt:.4f}")

print(f"\nTEST SET Performance:")
print(f"   Accuracy:  {test_acc_dt:.4f}")
print(f"   Precision: {test_precision_dt:.4f}")
print(f"   Recall:    {test_recall_dt:.4f}")
print(f"   F1-Score:  {test_f1_dt:.4f}")

overfit_gap_dt = train_acc_dt - test_acc_dt
print(f"\nOverfitting Analysis:")
print(f"   Train-Test Gap: {overfit_gap_dt:.4f}")
if overfit_gap_dt < 0.05:
    print(f"   Status: Good generalization (gap < 5%)")
elif overfit_gap_dt < 0.10:
    print(f"   Status: Slight overfitting (gap 5-10%)")
else:
    print(f"   Status: Significant overfitting (gap > 10%)")

print(f"\nDetailed Test Set Classification Report:")
print(classification_report(y_test, dt_test_pred, target_names=CLASSES))

## Section 10: Save Models

In [ ]:
print("\n" + "="*70)
print("SAVING TRAINED MODELS")
print("="*70)

svm_model_path = Path("svm_phase6_segmented.pkl")
dt_model_path = Path("dt_phase6_segmented.pkl")

joblib.dump(best_svm, svm_model_path)
joblib.dump(best_dt, dt_model_path)

print(f"\nModels saved successfully:")
print(f"   SVM model: {svm_model_path.resolve()}")
print(f"   Decision Tree model: {dt_model_path.resolve()}")

## Section 11: Model Comparison Summary

In [ ]:
print("\n" + "="*70)
print("FINAL MODEL COMPARISON - TRAIN vs TEST PERFORMANCE")
print("="*70)

comparison_data = {
    'Model': ['SVM', 'SVM', 'Decision Tree', 'Decision Tree'],
    'Dataset': ['Train', 'Test', 'Train', 'Test'],
    'Accuracy': [train_acc, test_acc, train_acc_dt, test_acc_dt],
    'Precision': [train_precision, test_precision, train_precision_dt, test_precision_dt],
    'Recall': [train_recall, test_recall, train_recall_dt, test_recall_dt],
    'F1-Score': [train_f1, test_f1, train_f1_dt, test_f1_dt]
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df[['Accuracy', 'Precision', 'Recall', 'F1-Score']] = comparison_df[['Accuracy', 'Precision', 'Recall', 'F1-Score']].round(4)

print("\nComprehensive Performance Table:")
print(comparison_df.to_string(index=False))

print("\n" + "="*70)
if test_acc > test_acc_dt:
    print(f"WINNER: SVM with Test Accuracy = {test_acc:.4f}")
    print(f"   (Decision Tree Test Accuracy = {test_acc_dt:.4f})")
    print(f"   Difference: {(test_acc - test_acc_dt)*100:.2f}%")
elif test_acc_dt > test_acc:
    print(f"WINNER: Decision Tree with Test Accuracy = {test_acc_dt:.4f}")
    print(f"   (SVM Test Accuracy = {test_acc:.4f})")
    print(f"   Difference: {(test_acc_dt - test_acc)*100:.2f}%")
else:
    print(f"TIE: Both models have Test Accuracy = {test_acc:.4f}")

print("="*70)

print(f"\nSummary Statistics:")
print(f"\n   SVM:")
print(f"   Best Parameters: {svm_random.best_params_}")
print(f"   Train Accuracy: {train_acc:.4f}")
print(f"   Test Accuracy:  {test_acc:.4f}")
print(f"   Generalization Gap: {(train_acc - test_acc)*100:.2f}%")

print(f"\n   Decision Tree:")
print(f"   Best Parameters: {dt_random.best_params_}")
print(f"   Train Accuracy: {train_acc_dt:.4f}")
print(f"   Test Accuracy:  {test_acc_dt:.4f}")
print(f"   Generalization Gap: {(train_acc_dt - test_acc_dt)*100:.2f}%")

print(f"\nPhase 6 Training Complete!")

## Section 12: Comparison with Phase 3 Models

In [ ]:
print("\n" + "="*70)
print("COMPARISON: Phase 3 (Full Images) vs Phase 6 (Segmented Images)")
print("="*70)

try:
    try:
        svm_phase3 = joblib.load("svm_tuned_loguniform_scaled.pkl")
        print("Loaded: svm_tuned_loguniform_scaled.pkl")
    except FileNotFoundError:
        svm_phase3 = joblib.load("svm_phase3.pkl")
        print("Loaded: svm_phase3.pkl")
    
    dt_phase3 = joblib.load("dt_phase3.pkl")
    
    df_phase3 = pd.read_csv("feature2.csv")
    
    df_phase3["label_numeric"] = df_phase3["class"].astype("category").cat.codes
    
    y_phase3 = df_phase3['label_numeric']
    X_phase3 = df_phase3.select_dtypes(include=["int64", "float64"]).drop(columns=['label', 'label_numeric'], errors='ignore')
    
    _, X_test_phase3, _, y_test_phase3 = train_test_split(
        X_phase3, y_phase3,
        test_size=0.3,
        random_state=42,
        stratify=y_phase3
    )
    
    svm_phase3_pred = svm_phase3.predict(X_test_phase3)
    dt_phase3_pred = dt_phase3.predict(X_test_phase3)
    
    svm_phase3_acc = accuracy_score(y_test_phase3, svm_phase3_pred)
    dt_phase3_acc = accuracy_score(y_test_phase3, dt_phase3_pred)
    
    print("\nPhase 3 (Full Images) Test Accuracy:")
    print(f"   SVM: {svm_phase3_acc:.4f}")
    print(f"   Decision Tree: {dt_phase3_acc:.4f}")
    
    print("\nPhase 6 (Segmented Images) Test Accuracy:")
    print(f"   SVM: {test_acc:.4f}")
    print(f"   Decision Tree: {test_acc_dt:.4f}")
    
    print("\nImprovement Analysis:")
    svm_improvement = test_acc - svm_phase3_acc
    dt_improvement = test_acc_dt - dt_phase3_acc
    
    print(f"   SVM: {svm_improvement:+.4f} ({svm_improvement*100:+.2f}%)")
    print(f"   Decision Tree: {dt_improvement:+.4f} ({dt_improvement*100:+.2f}%)")
    
    if svm_improvement > 0 and dt_improvement > 0:
        print("\nConclusion: Segmentation improved both models")
    elif svm_improvement > 0 or dt_improvement > 0:
        print("\nConclusion: Segmentation improved at least one model")
    else:
        print("\nConclusion: Full images performed better")
    
except Exception as e:
    print(f"\nCould not load Phase 3 models for comparison: {str(e)}")
    print("Skipping Phase 3 vs Phase 6 comparison")

print("="*70)